<a href="https://colab.research.google.com/github/Iqra411/lyrank-ML-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Iqra411/lyrank-ML-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

Lane: Refresh / Content Opportunity Scoring.
I'm picking this lane because it has a direct decision behind it — which of many underperforming pages a content reviewer should look at first — and because the starter pipeline already shows a learned ranking clearly beating a hand-written rule on this exact task (Precision@50 jumps from 0.240 with the baseline rules to 0.740 with a random forest). That gap is worth investigating further: it suggests there's real, learnable signal in observable metrics like impressions, clicks, position, and freshness that a simple rule misses. I want to spend the next 7 weeks understanding why the model beats the baseline, and turn that into an honest, ranked review queue rather than a black-box score.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

Question: Given a page's observed search and engagement signals over the last 90 days, which pages should a content reviewer look at first for a possible refresh?
   Unit of analysis: one content page (content_id), one row per page in the starter dataset.
   Decision this improves: which pages in a large content inventory get limited reviewer time and attention first.
   Who acts on it: a content strategist or SEO reviewer with capacity to manually review only a small number of pages per week (e.g. top 20–50).
   The action: the reviewer opens the top-ranked pages and decides whether to refresh, expand, protect, prune, or monitor each one — this notebook doesn't take the action itself, it prioritizes the queue.
   Cost of a wrong recommendation: a false positive wastes a reviewer's limited time on a page that wasn't actually worth refreshing — direct time cost, low severity. A false negative is worse: a real declining, high-traffic page never gets reviewed at all, and its visibility keeps eroding silently — this is why precision and recall both matter, but precision@K matters most given the reviewer's fixed capacity.
   Why data/ML can help at all: the starter pipeline already shows a hand-tuned rule identifies real signal (baseline ROC AUC 0.627) but a learned model captures meaningfully more of it (random forest ROC AUC 0.750, Precision@50 up from 0.240 to 0.740) — meaning the relationship between these signals and decline isn't fully captured by a simple linear rule, which is exactly the kind of pattern ML is suited to find.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/Iqra411/lyrank-ML-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print("Rows, columns:", df.shape)
print("\nAll column names:")
print(df.columns.tolist())

def find_col(keyword):
    matches = [c for c in df.columns if keyword.lower() in c.lower()]
    return matches[0] if matches else None

trend_col = find_col("trend")
impressions_col = find_col("impression")

print("\nDetected trend column:", trend_col)
print("Detected impressions column:", impressions_col)

# Number 1: how many pages are currently flagged as declining
if trend_col:
    decline_rate = (df[trend_col].astype(str).str.lower() == "down").mean()
    print(f"\nShare of pages currently trending down: {decline_rate:.1%}")

# Number 2: how many pages have real traffic (i.e. matter enough to prioritize)
if impressions_col:
    visible_pages = (df[impressions_col] >= 500).sum()
    print(f"Pages with >= 500 impressions: {visible_pages} of {len(df)}")

# Number 3: among visible pages, how many are ALSO declining
if trend_col and impressions_col:
    visible_and_declining = (
        (df[impressions_col] >= 500) &
        (df[trend_col].astype(str).str.lower() == "down")
    ).sum()
    print(f"Pages that are BOTH visible and declining: {visible_and_declining}")

Rows, columns: (30000, 44)

All column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Detected trend column: trend_direction
Detected impressions column: impressions_90d

Share of pages currently trending down: 54.2%
Pages with >= 500 impressions: 16726 of 30000
Pages

## 4. Careful words: what I can and can't claim

This work will be able to say: which pages, based on observed 90-day signals, look directionally similar to pages that historically declined — and how a learned ranking compares to a simple rule on this anonymized starter slice.
It will not be able to say: that any single page is guaranteed to decline, that refreshing a page will cause a recovery (that requires a real experiment, not this data), or anything about why Google's algorithm behaves a certain way. All results here are decision-support — a ranked list for a human reviewer — not automated causal claims. The starter pipeline numbers (Precision@50 = 0.740) come from a 30,000-row anonymized sample with client-holdout validation; they are not yet a benchmark on the full ~79M-row warehouse, and I'll need to re-earn that result if I move to the full release in later weeks.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [ ✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✅] No client names, URLs, or private queries anywhere
- [ ✅] My claims use careful words: observed, measured, directional, decision-support
- [ ✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.